# BFND: Supervised run guide (00)


## Phase 0: Setup

 (Runtime > Change runtime type > T4 GPU).

In [1]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

CUDA: True Tesla T4


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, shutil, zipfile
from pathlib import Path

ZIP = Path("/content/drive/MyDrive/Bangla-Multimodal-FakeNews.zip")
CODE_DIR = Path("/content/Bangla-Multimodal-FakeNews")
assert ZIP.exists(), "Code zip MyDrive-e nei. Latest zip upload korun."
shutil.rmtree(CODE_DIR, ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
os.chdir(CODE_DIR)
cfg = Path("configs/config.py").read_text(encoding="utf-8")
assert "TEXT_BACKBONES" in cfg and "LLM_BACKEND" in cfg, "Purono code. Latest zip upload korun."
print("cwd:", os.getcwd(), "| code version OK")

cwd: /content/Bangla-Multimodal-FakeNews | code version OK


In [4]:
!pip install -q "accelerate>=0.26" "transformers>=4.46" "sentence-transformers>=3.0" "rank-bm25>=0.2.2" joblib openpyxl requests

In [5]:
from configs.config import BASE, RAW_DIR, PROCESSED_DIR, FEATURE_DIR, LLM_BACKEND, LOCAL_LLM_MODEL
print("BASE:", BASE)
print("LLM backend:", LLM_BACKEND, "|", LOCAL_LLM_MODEL)
for p in [RAW_DIR, PROCESSED_DIR / "unified_split.csv", FEATURE_DIR / "meta.csv", FEATURE_DIR / "bert.npy", FEATURE_DIR / "clip_img.npy"]:
    print(("OK   " if p.exists() else "MISS ") + str(p))

BASE: /content/drive/MyDrive/BFND-data
LLM backend: local | Qwen/Qwen2.5-3B-Instruct
OK   /content/drive/MyDrive/BFND-data/data/raw
OK   /content/drive/MyDrive/BFND-data/data/processed/unified_split.csv
OK   /content/drive/MyDrive/BFND-data/features/meta.csv
OK   /content/drive/MyDrive/BFND-data/features/bert.npy
OK   /content/drive/MyDrive/BFND-data/features/clip_img.npy


## Phase 1: Cultural silver label


In [ ]:
!python scripts/05_build_cultural_silver.py

config.json: 100% 661/661 [00:00<00:00, 3.05MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 20.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 69.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 118MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 147MB/s]
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 101MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   9% 548M/6.17G [00:03<00:27, 207MB/s,  197MB/s  ]

Reconstructing (incomplete total...):  90% 5.55G/6.17G [00:46<00:05, 113MB/s,  137MB/s  ]
Reconstructing (incomplete total...):  96% 5.93G/6.17G [00:54<00:03, 71.9MB/s,  113MB/s  ]
Reconstructing (incomplete total...): 100% 6.17G/6.17G [00:57<00:00, 79.0MB/s, 71.9MB/s  ]

Fetching 2 files: 100% 2/2 [00:57<00:00, 28.81s/it]
Download complete: 100% 5.41G/5.41G [

### ⛔ 1A: Human check (40 row)



In [ ]:
!python scripts/05_build_cultural_silver.py

Loading weights: 100% 434/434 [00:16<00:00, 27.09it/s]
cultural_cache.jsonl: 3910 cached, 0 to compute
Silver labels: 3910 rows -> /content/drive/MyDrive/BFND-data/data/annotations/cultural_silver_labels.csv
Positive rate per feature: {'sarcasm': 0.0284, 'humor': 0.0317, 'local_reference': 0.3972, 'historical_reference': 0.0824}
Human check: 0 of 40 rows fully labelled
No filled rows yet; agreement not computed.


In [ ]:
import json
import pandas as pd
from configs.config import RESULTS_DIR, CULTURAL_SILVER_CSV, CULTURAL_CHECK_CSV

if not CULTURAL_SILVER_CSV.exists():
    print("MISSING:", CULTURAL_SILVER_CSV)
    print("Silver label file ekhono toiri hoyni. Ei karon-er ekta:")
    print("  (a) Upore 'Phase 1' er script cell (05) shesh hoyni: cell-ta ekhono chalchhe ba thamiyeche, shesh na hole abar chalan (cache thakle resume hobe)")
    print("  (b) 05 cell-e error hoyeche: oi cell-er output (error) paste korun")
    print("Ei cell-ta 05 shesh holeo abar chalan।")
    raise SystemExit
silver = pd.read_csv(CULTURAL_SILVER_CSV)
print("Silver rows:", len(silver))
print("Positive rate per feature (LLM):")
print(silver.drop(columns="sample_id").mean().round(3).to_string())

agr_path = RESULTS_DIR / "cultural_agreement.json"
if not agr_path.exists():
    print("\nAgreement nei. Human check sheet ta bharun, tarpor is cell abar chalan.")
else:
    report = json.loads(agr_path.read_text(encoding="utf-8"))
    print("\nAgreement with human labels (proposed thresholds: PASS >= 0.6, REVIEW 0.4-0.6, FAIL < 0.4):")
    for name, r in report.items():
        k = r.get("kappa")
        if k is None:
            status = "NO DATA (human has not filled this feature)"
        elif k >= 0.6:
            status = "PASS"
        elif k >= 0.4:
            status = "REVIEW"
        else:
            status = "FAIL: do not use this feature"
        print(f"  {name:<22} n={r.get('n')}  agreement={r.get('agreement')}  kappa={k}  -> {status}")


Silver rows: 3910
Positive rate per feature (LLM):
sarcasm                 0.028
humor                   0.032
local_reference         0.397
historical_reference    0.082

Agreement nei. Human check sheet ta bharun, tarpor is cell abar chalan.


## Phase 2: Wikipedia evidence।

In [ ]:
import time, requests
from src.wiki_evidence import USER_AGENT
from configs.config import WIKI_API, CACHE_DIR

t = time.time()
try:
    r = requests.get(WIKI_API,
                     params={"action": "query", "list": "search", "srsearch": "বাংলাদেশ", "format": "json"},
                     headers={"User-Agent": USER_AGENT}, timeout=15)
    print("status:", r.status_code, "| seconds:", round(time.time() - t, 2))
    print(r.text[:300])
except Exception as e:
    print("ERROR:", type(e).__name__, str(e)[:300], "| seconds:", round(time.time() - t, 2))

print("wiki_search_cache lines:", sum(1 for _ in open(CACHE_DIR / "wiki_search_cache.jsonl", encoding="utf-8")))

status: 200 | seconds: 0.29
{"batchcomplete":"","continue":{"sroffset":10,"continue":"-||"},"query":{"searchinfo":{"totalhits":45398},"search":[{"ns":0,"title":"\u09ac\u09be\u0982\u09b2\u09be\u09a6\u09c7\u09b6","pageid":822,"size":492520,"wordcount":21123,"snippet":"<span class=\"searchmatch\">\u09ac\u09be\u0982\u09b2\u09be\u0
wiki_search_cache lines: 98


In [ ]:
!python scripts/06_build_evidence_wiki.py

All queries are cached: LLM model not loaded.
query_cache.jsonl: 3910 cached, 0 to compute
wiki_search_cache.jsonl: 98 cached, 3623 to compute
  20/3623
  40/3623
  60/3623
  80/3623
  100/3623
  120/3623
  140/3623
  160/3623
  180/3623
  200/3623
  220/3623
  240/3623
  260/3623
  280/3623
  300/3623
  320/3623
  340/3623
  360/3623
  380/3623
  400/3623
  420/3623
  440/3623
  460/3623
  480/3623
  500/3623
  520/3623
  540/3623
  560/3623
  580/3623
  600/3623
  620/3623
  640/3623
  660/3623
  680/3623
  700/3623
  720/3623
  740/3623
  760/3623
  780/3623
  800/3623
  820/3623
  840/3623
  860/3623
  880/3623
  900/3623
  920/3623
  940/3623
  960/3623
  980/3623
  1000/3623
  1020/3623
  1040/3623
  1060/3623
  1080/3623
  1100/3623
  1120/3623
  1140/3623
  1160/3623
  1180/3623
  1200/3623
  1220/3623
  1240/3623
  1260/3623
  1280/3623
  1300/3623
  1320/3623
  1340/3623
  1360/3623
  1380/3623
  1400/3623
  1420/3623
  1440/3623
  1460/3623
  1480/3623
  1500/3623
  1520/362

### ⛔ 2A: Relevance check (apnar kaj, 40 row)

1. Drive: `MyDrive/BFND-data/annotations/evidence_relevance_check.csv`
2. `text`, `query`, `wiki_title` dekhe `relevant_0_or_1`-e 1 (Wikipedia article ta post-er fact-er sathe sambhab) ba 0 din।
3. Row-er order badlaben na। Save korun।


In [ ]:
import json
import pandas as pd
from configs.config import RESULTS_DIR, EVIDENCE_RELEVANCE_CHECK_CSV

from configs.config import EVIDENCE_CSV
if not (RESULTS_DIR / "evidence_build_report.json").exists():
    print("MISSING: evidence_build_report.json. Upore 'Phase 2' script cell (06) shesh hoyni ba error diyeche.")
    print("Shei cell-er output paste korun; shesh na hole abar chalan।")
    raise SystemExit
rep = json.loads((RESULTS_DIR / "evidence_build_report.json").read_text(encoding="utf-8"))
print(json.dumps(rep, ensure_ascii=False, indent=2))
cov = rep["coverage_share"]
cov_status = "PASS" if cov >= 0.5 else ("REVIEW" if cov >= 0.3 else "WEAK: E5 gain will likely be small")
print(f"\nCoverage {cov:.3f} -> {cov_status}   (proposed: PASS >= 0.5, REVIEW 0.3-0.5)")

check = pd.read_csv(EVIDENCE_RELEVANCE_CHECK_CSV)
filled = check["relevant_0_or_1"].dropna()
if len(filled) == 0:
    print("Relevance nei. Sheet ta bharun, tarpor is cell abar chalan.")
else:
    rate = filled.astype(int).mean()
    rel_status = "PASS" if rate >= 0.6 else ("REVIEW" if rate >= 0.4 else "FAIL")
    print(f"Relevance {rate:.3f} on {len(filled)} rows -> {rel_status}   (proposed: PASS >= 0.6, REVIEW 0.4-0.6)")


{
  "posts": 3910,
  "posts_without_query": 91,
  "query_parse_failures": 0,
  "wiki_request_failures": 1680,
  "unique_queries": 3721,
  "posts_with_at_least_one_article": 1437,
  "coverage_share": 0.3675,
  "articles": 2085,
  "passages": 5580,
  "llm_model": "claude-haiku-5-5",
  "evidence_source": "Bangla Wikipedia lead sections; query from post text only",
  "chunk_chars": 500,
  "search_limit": 2
}

Coverage 0.367 -> REVIEW   (proposed: PASS >= 0.5, REVIEW 0.3-0.5)
Relevance nei. Sheet ta bharun, tarpor is cell abar chalan.


⛔ **STOP 2:** Coverage ar relevance output paste korun। Relevance FAIL hole E5 chalanor age ami bolbo।

## Phase 3: Feature refresh

**Ki hobe:** script 02 abar chalabe। Eta BanglaBERT ar CLIP feature abar banabe (kichhu minute), ar cultural.npy ar evidence.npy toiri korbe।
Cultural source: manual file nei, tai `llm_silver` use hobe (`results/cultural_source.json`-e lekha)।

In [6]:
!python scripts/02_extract_features.py

config.json: 100% 586/586 [00:00<00:00, 2.49MB/s]
tokenizer_config.json: 100% 119/119 [00:00<00:00, 507kB/s]
vocab.txt: 100% 528k/528k [00:00<00:00, 39.4MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 697kB/s]

pytorch_model.bin: downloading bytes:  86% 379M/443M [00:02<00:00, 254MB/s, 31.4MB/s  ]
pytorch_model.bin: downloading bytes:  95% 419M/443M [00:03<00:00, 234MB/s, 35.8MB/s  ]
pytorch_model.bin: downloading bytes: 100% 419M/419M [00:04<00:00, 101MB/s, 36.8MB/s  ]
pytorch_model.bin: reconstructing file: 100% 443M/443M [00:04<00:00, 106MB/s, 39.8MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 24742.56it/s]
[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictio

In [7]:
import json
import numpy as np
from configs.config import FEATURE_DIR, RESULTS_DIR

for name in ["bert", "clip_img", "clip_txt", "sim", "has_image", "cultural", "evidence"]:
    p = FEATURE_DIR / f"{name}.npy"
    print(f"{name:<10}", np.load(p).shape if p.exists() else "MISSING")
src = RESULTS_DIR / "cultural_source.json"
print("\ncultural source:", json.loads(src.read_text()) if src.exists() else "not built")
guard = RESULTS_DIR / "retrieval_report.json"
print("retrieval report:", json.loads(guard.read_text()) if guard.exists() else "not built")

bert       (3910, 768)
clip_img   (3910, 512)
clip_txt   (3910, 512)
sim        (3910, 1)
has_image  (3910, 1)
cultural   (3910, 4)
evidence   (3910, 3)

cultural source: {'source': 'llm_silver', 'file': 'cultural_silver_labels.csv', 'features': ['sarcasm', 'humor', 'local_reference', 'historical_reference']}
retrieval report: {'label_source_guard': 'not_applicable: BTMD has no source URL; evidence is Wikipedia built from claim text only', 'dropped_label_source': 0, 'evidence_used': 10448, 'recall_at_k_test': None, 'n_test_with_gold': 0}


⛔ **STOP 3:** Shape table ar retrieval report paste korun।

## Phase 4: Backbone features (mBERT, XLM-R, ViT, Swin)

**Ki hobe:** `bert.npy` ar `clip_img.npy` reuse hobe। Baki 4-ti feature notun kore banabe।

In [ ]:
!python scripts/07_extract_backbones.py

[REUSE] text banglabert: bert.npy (3910, 768)
config.json: 100% 625/625 [00:00<00:00, 2.61MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 51.8kB/s]
vocab.txt: 100% 996k/996k [00:00<00:00, 16.3MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 5.59MB/s]

model.safetensors: downloading bytes:  53% 380M/714M [00:01<00:01, 286MB/s, 33.4MB/s  ]
model.safetensors: downloading bytes:  65% 463M/714M [00:02<00:00, 282MB/s, 39.1MB/s  ]
model.safetensors: downloading bytes:  76% 545M/714M [00:02<00:00, 239MB/s, 45.6MB/s  ]
model.safetensors: downloading bytes:  94% 668M/714M [00:02<00:00, 322MB/s, 54.4MB/s  ]
model.safetensors: reconstructing file:  66% 469M/714M [00:02<00:01, 187MB/s, 36.6MB/s  ]
model.safetensors: downloading bytes: 100% 676M/676M [00:03<00:00, 174MB/s, 57.9MB/s  ]
model.safetensors: reconstructing file: 100% 714M/714M [00:03<00:00, 184MB/s, 61.0MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 4088.60it/s]
[transformers] BertModel LOAD REPORT from: bert-base-mult

## Phase 5: Selection grid

In [ ]:
!python scripts/08_model_selection.py

Configurations: 35 x seeds [42, 43, 44] = 105 runs
[  1/105] text:banglabert                          seed 42: val F1 0.6908 | test F1 0.5880 acc 0.7568
[  2/105] text:banglabert                          seed 43: val F1 0.6983 | test F1 0.6463 acc 0.7789
[  3/105] text:banglabert                          seed 44: val F1 0.6860 | test F1 0.6471 acc 0.7874
[  4/105] text:mbert                               seed 42: val F1 0.6346 | test F1 0.5971 acc 0.6956
[  5/105] text:mbert                               seed 43: val F1 0.6388 | test F1 0.6273 acc 0.7279
[  6/105] text:mbert                               seed 44: val F1 0.6271 | test F1 0.6427 acc 0.7415
[  7/105] text:xlmr                                seed 42: val F1 0.6935 | test F1 0.6312 acc 0.7381
[  8/105] text:xlmr                                seed 43: val F1 0.6890 | test F1 0.6723 acc 0.7585
[  9/105] text:xlmr                                seed 44: val F1 0.6962 | test F1 0.6295 acc 0.7449
[ 10/105] image:clip           

In [ ]:
import json
import numpy as np
from configs.config import SELECTION_DIR, LABELS_3
from src.selection import format_table, confusion_text

if not (SELECTION_DIR / "summary.json").exists():
    print("MISSING: results/selection/summary.json. Phase 5 script cell (08) shesh hoyni ba error diyeche; output paste korun।")
    raise SystemExit
summary = json.loads((SELECTION_DIR / "summary.json").read_text(encoding="utf-8"))
cols = [("val_macro_f1", "Val F1"), ("test_accuracy", "Test Acc"), ("test_macro_precision", "Test Prec"),
        ("test_macro_recall", "Test Recall"), ("test_macro_f1", "Test F1")]
for kind, title in [("text", "TEXT ONLY"), ("image", "IMAGE ONLY"), ("control", "CONTROLS"), ("fusion", "FUSION")]:
    names = sorted([n for n, r in summary.items() if r["kind"] == kind], key=lambda n: -summary[n]["val_macro_f1_mean"])
    print("=" * 110); print(title); print(format_table(summary, names, cols)); print()
for name in sorted(summary, key=lambda n: (summary[n]["kind"], -summary[n]["val_macro_f1_mean"])):
    print("=" * 80); print(name)
    print(confusion_text(np.array(summary[name]["test_confusion_mean"]), LABELS_3)); print()
best = json.loads((SELECTION_DIR / "best_config.json").read_text(encoding="utf-8"))
print("WINNERS (validation-only):", best)

TEXT ONLY
config                                                Val F1          Test Acc         Test Prec       Test Recall           Test F1
------------------------------------------------------------------------------------------------------------------------------------
text:xlmr                                     0.6929+-0.0030    0.7472+-0.0085    0.6501+-0.0209    0.6408+-0.0197    0.6443+-0.0198
text:banglabert                               0.6917+-0.0050    0.7744+-0.0129    0.6278+-0.0244    0.6295+-0.0276    0.6271+-0.0277
text:mbert                                    0.6335+-0.0048    0.7217+-0.0193    0.6309+-0.0329    0.6191+-0.0069    0.6224+-0.0190

IMAGE ONLY
config                                                Val F1          Test Acc         Test Prec       Test Recall           Test F1
------------------------------------------------------------------------------------------------------------------------------------
image:clip                                    0

## Phase 6

In [8]:
!python scripts/04_train_fusion.py

E0_has_image_only seed=42: acc=0.2245 macroF1=0.2023
E0_has_image_only seed=43: acc=0.2245 macroF1=0.2023
E0_has_image_only seed=44: acc=0.2245 macroF1=0.2023
E1f_frozen_banglabert_mlp seed=42: acc=0.7568 macroF1=0.5880
E1f_frozen_banglabert_mlp seed=43: acc=0.7789 macroF1=0.6463
E1f_frozen_banglabert_mlp seed=44: acc=0.7874 macroF1=0.6471
E2_clip_image_only seed=42: acc=0.5952 macroF1=0.5335
E2_clip_image_only seed=43: acc=0.6071 macroF1=0.5557
E2_clip_image_only seed=44: acc=0.6003 macroF1=0.5347
E3_banglabert_clip seed=42: acc=0.8333 macroF1=0.7266
E3_banglabert_clip seed=43: acc=0.8333 macroF1=0.7426
E3_banglabert_clip seed=44: acc=0.8452 macroF1=0.7652
E5_plus_evidence seed=42: acc=0.8282 macroF1=0.7240
E5_plus_evidence seed=43: acc=0.8214 macroF1=0.7248
E5_plus_evidence seed=44: acc=0.8282 macroF1=0.7238
E6: {'min_confidence_from_validation': 0.7224954962730408, 'target_coverage': 0.85, 'final_label_counts_test': {'Insufficient': 376, 'Authentic': 138, 'Misleading': 68, 'Satire':

In [9]:
from configs.config import RESULTS_DIR
print((RESULTS_DIR / "summary.md").read_text(encoding="utf-8"))

| Experiment | Features | Accuracy (mean ± std) | Macro-F1 (mean ± std) |
|---|---|---|---|
| E0_has_image_only | has_image | 0.2245 ± 0.0000 | 0.2023 ± 0.0000 |
| E1f_frozen_banglabert_mlp | bert | 0.7744 ± 0.0129 | 0.6271 ± 0.0277 |
| E2_clip_image_only | clip_img | 0.6009 ± 0.0049 | 0.5413 ± 0.0102 |
| E3_banglabert_clip | bert, clip_img, clip_txt, sim | 0.8373 ± 0.0056 | 0.7448 ± 0.0158 |
| E5_plus_evidence | bert, clip_img, clip_txt, sim, evidence | 0.8260 ± 0.0032 | 0.7242 ± 0.0004 |

